# Tuần 3: LLM Few-shot ICL + Aspect-aware RAG
## ABSA VLSP 2018 Hotel — NLP Course HUST

**Mục tiêu:**
- Tầng 2: Few-shot ICL với random examples (GPT-4o-mini + Gemini 1.5 Flash)
- Tầng 3: Aspect-aware RAG với semantic retrieval
- So sánh: ICL vs RAG vs PhoBERT (Tầng 1)

**Chạy trên:** Google Colab hoặc Kaggle (CPU đủ dùng, không cần GPU)

In [ ]:
import os
!pip install -q openai google-generativeai sentence-transformers faiss-cpu

# Set API keys — điền vào đây
OPENAI_API_KEY = "sk-..."      # hoặc để "" nếu không có
GEMINI_API_KEY = "AIza..."     # hoặc để "" nếu không có

api_keys = {}
if OPENAI_API_KEY and OPENAI_API_KEY != "sk-...": api_keys["openai"] = OPENAI_API_KEY
if GEMINI_API_KEY and GEMINI_API_KEY != "AIza...": api_keys["gemini"] = GEMINI_API_KEY
print(f"Providers available: {list(api_keys.keys())}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/absa-vlsp2018-hotel")
import sys
sys.path.insert(0, "code/week1")
sys.path.insert(0, "code/week3")
print("Ready ✓")

In [ ]:
import pandas as pd
from prompts import build_prompt, parse_llm_output
from llm_client import LLMClient

train_df = pd.read_csv("data/train_preprocessed.csv")
test_df  = pd.read_csv("data/test_preprocessed.csv")

# Test 1 review
client = LLMClient(provider="openai", api_key=api_keys["openai"])
sample = test_df.iloc[0]["processed_review"]
examples = [{"review": train_df.iloc[i]["processed_review"],
             "labels": {}} for i in range(2)]  # dummy examples

messages = build_prompt(sample, examples)
response = client.complete(messages)
result = parse_llm_output(response)

print(f"Review: {sample[:100]}")
print(f"Raw output: {response[:200]}")
print(f"Parsed: {result}")

In [ ]:
# max_samples=100 để test nhanh (~$0.05)
# max_samples=None để chạy full 600 test reviews (~$0.50)
from run_tier2 import main as run_tier2
run_tier2(api_keys=api_keys, max_samples=100)

In [ ]:
from run_tier3 import main as run_tier3
run_tier3(api_keys=api_keys, max_samples=100)

In [ ]:
from compare_results import generate_comparison_table
report = generate_comparison_table()
print(report)

In [ ]:
import json, matplotlib.pyplot as plt

# Load tất cả results
providers = [p for p in ["openai", "gemini"] if p in api_keys]
k_values  = [2, 4, 8]

fig, axes = plt.subplots(1, len(providers), figsize=(14, 5))
if len(providers) == 1:
    axes = [axes]

for ax, provider in zip(axes, providers):
    icl_f1s, rag_f1s = [], []
    for k in k_values:
        icl = json.load(open(f"outputs/results/tier2_{provider}_k{k}_metrics.json"))
        rag = json.load(open(f"outputs/results/tier3_{provider}_k{k}_metrics.json"))
        icl_f1s.append(icl["macro_combined_f1"])
        rag_f1s.append(rag["macro_combined_f1"])

    ax.plot(k_values, icl_f1s, "o-", color="steelblue",  label="ICL (random)", lw=2)
    ax.plot(k_values, rag_f1s, "o-", color="darkorange", label="RAG (retrieval)", lw=2)
    ax.axhline(y=__WEEK2_COMBINED_F1__, color="green", ls="--",
               label=f"PhoBERT ({__WEEK2_COMBINED_F1__:.4f})", alpha=0.7)
    ax.axhline(y=0.7732, color="red", ls=":", label="SOTA (0.7732)", alpha=0.5)
    ax.set(title=f"{provider.upper()} — ICL vs RAG",
           xlabel="k (số examples)", ylabel="Combined F1")
    ax.set_xticks(k_values)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/week3_ablation.png", dpi=150, bbox_inches="tight")
plt.show()